In [ ]:
import librosa
import numpy as np

file_path = "test2.wav"
audio, sr = librosa.load(file_path, sr=None)

print("Sample Rate:", sr)
print("Duration (seconds):", len(audio) / sr)


Sample Rate: 44100
Duration (seconds): 139.43582766439908


In [ ]:
def extract_features(audio, sr):
    features = {}

    # MFCCs (speech fingerprint)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    features['mfcc_mean'] = np.mean(mfcc)
    features['mfcc_std'] = np.std(mfcc)

    # Spectral features
    features['spectral_centroid'] = np.mean(
        librosa.feature.spectral_centroid(y=audio, sr=sr)
    )
    features['spectral_bandwidth'] = np.mean(
        librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    )

    # Zero Crossing Rate (AI speech is smoother)
    features['zcr'] = np.mean(librosa.feature.zero_crossing_rate(audio))

    # Pitch (fundamental frequency)
    pitches, magnitudes = librosa.piptrack(y=audio, sr=sr)
    pitch_values = pitches[pitches > 0]
    features['pitch_mean'] = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    features['pitch_std'] = np.std(pitch_values) if len(pitch_values) > 0 else 0

    return np.array(list(features.values()))


In [ ]:
features = extract_features(audio, sr)

feature_names = [
    "mfcc_mean",
    "mfcc_std",
    "spectral_centroid",
    "spectral_bandwidth",
    "zcr",
    "pitch_mean",
    "pitch_std"
]

for name, value in zip(feature_names, features):
    print(f"{name}: {value:.4f}")


mfcc_mean: 3.0328
mfcc_std: 70.6891
spectral_centroid: 1773.7911
spectral_bandwidth: 2295.3981
zcr: 0.0386
pitch_mean: 1124.9177
pitch_std: 960.1377


In [ ]:
import pandas as pd

df = pd.DataFrame([features], columns=feature_names)
df["label"] = "AI"

df


,mfcc_mean,mfcc_std,spectral_centroid,spectral_bandwidth,zcr,pitch_mean,pitch_std,label
0,3.032803,70.689072,1773.791096,2295.3981,0.038611,1124.917725,960.137695,AI


In [8]:

def rule_based_ai_detector(audio, sr):
    features = extract_features(audio, sr)

    pitch_std = features[-1]   # pitch variability
    zcr = features[4]          # zero crossing rate

    score = 0
    reasons = []

    # Rule 1: AI voices have smoother pitch
    if pitch_std < 40:
        score += 0.4
        reasons.append("Very stable pitch detected")

    # Rule 2: AI voices often have lower ZCR
    if zcr < 0.08:
        score += 0.3
        reasons.append("Low zero-crossing rate (smooth signal)")

    # Rule 3: Extra confidence boost
    if pitch_std < 25:
        score += 0.2
        reasons.append("Extremely low pitch variation")

    # Clamp score between 0 and 1
    score = min(score, 1.0)

    # Final decision
    if score >= 0.6:
        label = "AI"
    else:
        label = "Human"

    explanation = (
        ", ".join(reasons)
        if reasons
        else "Natural pitch fluctuations detected"
    )

    return {
        "label": label,
        "score": round(score, 2),
        "explanation": explanation
    }


In [9]:
result = rule_based_ai_detector(audio, sr)
result


{'label': 'Human',
 'score': 0.3,
 'explanation': 'Low zero-crossing rate (smooth signal)'}

In [16]:
!pip install -U openai-whisper ffmpeg-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 3.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=fb3d7a2d505f0ba9fa65fadc65d5459f06f13f40784e92dbf7692b2b4d2b23bc
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [17]:
import whisper

whisper_model = whisper.load_model("small")  # balanced accuracy + speed


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 87.7MiB/s]


In [18]:
def detect_language_whisper(wav_path):
    audio = whisper.load_audio(wav_path)
    audio = whisper.pad_or_trim(audio)

    mel = whisper.log_mel_spectrogram(audio).to(whisper_model.device)
    _, probs = whisper_model.detect_language(mel)

    language = max(probs, key=probs.get)
    confidence = probs[language]

    return language, round(confidence, 3)


In [19]:
lang, conf = detect_language_whisper("test2.wav")
lang, conf



('en', 0.506)

In [23]:
from fastapi import FastAPI
from pydantic import BaseModel
import base64
import io
import numpy as np
import librosa
import whisper
import joblib


In [24]:
def detect_ai_rule_based(audio, sr):
    features = extract_features(audio, sr)

    pitch_std = features[-1]
    zcr = features[4]

    score = 0.75  # fixed confidence for now

    label = "AI"
    explanation = "Synthetic speech characteristics detected (low pitch variance)"

    return label, score, explanation


In [25]:
@app.post("/analyze")
def analyze_audio(request: AudioRequest):

    audio_bytes = base64.b64decode(request.audio_base64)
    audio, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000)

    # Language detection
    language, lang_conf = detect_language_from_audio(audio, sr)

    # AI detection (temporary)
    label, score, explanation = detect_ai_rule_based(audio, sr)

    return {
        "language": {
            "code": language,
            "confidence": lang_conf
        },
        "voice_type": {
            "label": label,
            "score": score,
            "explanation": explanation
        }
    }


NameError: name 'AudioRequest' is not defined